# **Asistente-EEG: asistente educativo para el análisis de señales EEG**

Asistente-EEG es un asistente basado en inteligencia artificial diseñado para facilitar la exploración y comprensión de señales electroencefalográficas (EEG). Permite analizar señales, consultar conceptos relacionados con EEG y comparar la potencia relativa de sus bandas de frecuencia.

El proyecto integra herramientas de procesamiento de señales con un agente capaz de seleccionar y ejecutar las funciones disponibles según las necesidades del usuario.

---

### **Configuración de la API Key (NVIDIA NIM)**
Para ejecutar el asistente, necesitas una clave de API de NVIDIA. Puedes obtenerla gratuitamente registrándote e ingresando a la plataforma de **[NVIDIA NIM / Build](https://build.nvidia.com)** para generar tu credencial.

* **En Google Colab:** Abre el panel izquierdo, haz clic en el ícono de **Secretos** (la llave 🔑), crea un nuevo secreto llamado `NVIDIA_API_KEY`, pega tu clave y activa el acceso al cuaderno.
* **En entorno local:** Crea un archivo `.env` en la misma carpeta del proyecto y añade la línea `NVIDIA_API_KEY="tu_clave_aqui"`, o configúrala como variable de entorno en tu sistema.

In [1]:
# Instalaciones requeridas
%pip install openai httpx numpy scipy matplotlib gradio mne python-dotenv --quiet

In [2]:
#Configuración inicial y librerías
import os
import re
import json
import numpy as np
from scipy.signal import welch
from scipy.integrate import trapezoid
import mne
from openai import OpenAI
import gradio as gr
import matplotlib.pyplot as plt
import unicodedata

# ── CONFIGURACIÓN DE LA API KEY ──────────────────────────────────────────
# Intenta obtener la clave desde los secretos de Google Colab (entorno cloud)
# o recurre a un archivo .env local si se ejecuta fuera de Colab.
try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY", "")

if not NVIDIA_API_KEY:
    raise ValueError(
        "No se encontró NVIDIA_API_KEY. Configúrala en los secretos de Colab "
        "o como variable de entorno local."
    )

# ── INICIALIZACIÓN DEL CLIENTE NVIDIA NIM / OPENAI ───────────────────────
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY
)

MODEL = "nvidia/nemotron-3.5-super-vl-120b-a12b"

print("Entorno configurado correctamente.")

Entorno configurado correctamente.


In [3]:
# ── CREACIÓN DEL DIRECTORIO DE LA BASE DE CONOCIMIENTO ───────────────────
os.makedirs("knowledge_base", exist_ok=True)

# ── DEFINICIÓN DEL DICCIONARIO DE CONCEPTOS CIENTÍFICOS Y EDUCATIVOS ─────
# Estructura que alimenta la herramienta de consulta del agente (basada en literatura
# de la American Epilepsy Society y procesamiento espectral).
conceptos_data = {
    "meta": {
        "fuente_principal": (
            "Electroencephalography: An Introductory Text and Atlas "
            "(American Epilepsy Society) y literatura de procesamiento espectral EEG."
        ),
        "alcance": (
            "Educativo. Se limita a las bandas Delta, Theta, Alpha y Beta. "
            "Las asociaciones fisiológicas son generales y dependen del contexto."
        )
    },
    "bandas": {
        "delta": {
            "rango_hz": "1–4 Hz (convención operativa del análisis)",
            "caracteristicas_neurofisiologicas": (
                "Actividad de baja frecuencia. En el EEG puede reflejar actividad "
                "cortical sincronizada y contribuciones de redes distribuidas; "
                "no debe atribuirse a un único generador."
            ),
            "contexto_fisiologico": (
                "Es prominente durante el sueño N3 (sueño no REM de ondas lentas). "
                "En adultos despiertos suele tener menor representación, aunque "
                "su distribución depende del estado y de la región registrada."
            ),
            "consideraciones_interpretacion": (
                "La potencia delta en una señal aislada no permite establecer "
                "somnolencia, lesión ni diagnóstico alguno. Los movimientos y "
                "artefactos oculares también pueden aumentar la potencia de baja frecuencia."
            )
        },
        "theta": {
            "rango_hz": "4–8 Hz (convención operativa del análisis)",
            "caracteristicas_neurofisiologicas": (
                "Oscilaciones de frecuencia intermedia cuya expresión y significado "
                "dependen de la región cerebral, la tarea y el estado conductual."
            ),
            "contexto_fisiologico": (
                "Puede observarse durante somnolencia y transiciones hacia el sueño. "
                "También se estudia en procesos de memoria, control cognitivo y otras "
                "tareas; estas asociaciones no son exclusivas ni universales."
            ),
            "consideraciones_interpretacion": (
                "No es un marcador exclusivo de meditación, fatiga o memoria. "
                "La interpretación requiere contexto experimental y ubicación del electrodo."
            )
        },
        "alpha": {
            "rango_hz": "8–13 Hz (convención operativa del análisis)",
            "caracteristicas_neurofisiologicas": (
                "Ritmo especialmente visible en regiones posteriores del cuero cabelludo "
                "en adultos despiertos y relajados, con los ojos cerrados."
            ),
            "contexto_fisiologico": (
                "La actividad alpha posterior suele atenuarse al abrir los ojos "
                "o al realizar tareas que involucran procesamiento visual, fenómeno "
                "conocido como reactividad alpha."
            ),
            "consideraciones_interpretacion": (
                "Una mayor potencia relativa alpha no demuestra por sí sola calma, "
                "bienestar ni un estado psicológico determinado."
            )
        },
        "beta": {
            "rango_hz": "13–30 Hz (convención operativa del análisis)",
            "caracteristicas_neurofisiologicas": (
                "Actividad de frecuencia relativamente alta que puede observarse "
                "en distintas regiones. La actividad beta sensorimotora se estudia "
                "en relación con la preparación y ejecución del movimiento."
            ),
            "contexto_fisiologico": (
                "Puede variar con el estado de alerta, las tareas cognitivas y la "
                "actividad sensorimotora. No corresponde a un único proceso mental."
            ),
            "consideraciones_interpretacion": (
                "La actividad electromiográfica puede contaminar notablemente el "
                "rango beta y elevar su potencia estimada."
            )
        }
    },
    "artefactos": {
        "descripcion": (
            "Señales no cerebrales o interferencias instrumentales que se superponen "
            "al EEG y pueden modificar su espectro."
        ),
        "tipos_frecuentes": [
            (
                "Oculares (EOG): parpadeos y movimientos oculares producen "
                "deflexiones de gran amplitud, especialmente en electrodos frontales, "
                "con contribución importante a bajas frecuencias."
            ),
            (
                "Musculares (EMG): la actividad de músculos faciales, mandibulares "
                "y cervicales puede introducir potencia de alta frecuencia, incluida "
                "la banda Beta."
            ),
            (
                "Instrumentales y de red: mal contacto de electrodos, movimiento "
                "de cables e interferencia eléctrica de 50 o 60 Hz."
            )
        ]
    },
    "limitaciones": {
        "alcance_educativo": (
            "El sistema analiza señales sintéticas o archivos EDF con fines "
            "estrictamente pedagógicos."
        ),
        "restricciones_clinicas": (
            "El análisis espectral de un canal, sin contexto clínico ni revisión "
            "especializada, no permite diagnosticar epilepsia, trastornos del sueño "
            "ni otras patologías."
        ),
        "limitaciones_tecnicas": (
            "Las señales sintéticas no reproducen toda la complejidad, variabilidad "
            "temporal, topografía multicanal ni diversidad de artefactos de un EEG real. "
            "La potencia relativa depende del intervalo frecuencial y de los parámetros "
            "del método espectral."
        )
    }
}

# ── SERIALIZACIÓN Y GUARDADO EN JSON ─────────────────────────────────────
with open("knowledge_base/conceptos_eeg.json", "w", encoding="utf-8") as f:
    json.dump(conceptos_data, f, ensure_ascii=False, indent=2)

print("Base de conocimiento EEG creada correctamente.")

Base de conocimiento EEG creada correctamente.


# **Fundamentos del análisis EEG**

## **Bandas de frecuencia y análisis espectral**

El electroencefalograma (EEG) registra la actividad eléctrica cerebral en el dominio del tiempo (amplitud en $\mu\text{V}$ frente al tiempo). Para identificar los ritmos oscilatorios que componen la señal, es necesario transformarla al dominio de la frecuencia mediante un análisis espectral.  

### **1. Densidad Espectral de Potencia (Método de Welch)**
En lugar de aplicar una única Transformada de Fourier sobre toda la señal —que produce espectros ruidosos ante señales no estacionarias—, este sistema utiliza el **método de Welch** (`scipy.signal.welch`) para estimar la Densidad Espectral de Potencia ($\text{PSD}$, en $\mu\text{V}^2/\text{Hz}$):
1. Divide la señal en segmentos con un solapamiento del $50\%$ (`noverlap = nperseg // 2`).
2. Aplica una ventana de **Hann** (`window="hann"`) a cada tramo para evitar fugas espectrales.
3. Calcula la **Transformada Rápida de Fourier (FFT)** de cada segmento y promedia los espectros resultantes para reducir el ruido.

### **2. Bandas de frecuencia y potencia relativa**
La potencia absoluta de cada banda ($\mu\text{V}^2$) se calcula integrando numéricamente la $\text{PSD}$ mediante la **regla del trapecio** (`scipy.integrate.trapezoid`). La potencia relativa expresa el porcentaje respecto a la potencia total entre $1$ y $30\text{ Hz}$:

$$P_{\text{relativa}} (\%) = \left( \frac{P_{\text{banda}}}{P_{\text{total } (1-30\text{ Hz})}} \right) \times 100$$
* **Delta ($1 - 4\text{ Hz}$):** Ondas lentas predominantes en el sueño profundo (N3).
* **Theta ($4 - 8\text{ Hz}$):** Asociada a somnolencia, transición al sueño y procesos cognitivos.
* **Alpha ($8 - 13\text{ Hz}$):** Característica de regiones posteriores en vigilia relajada con ojos cerrados.
* **Beta ($13 - 30\text{ Hz}$):** Vinculada al estado de alerta, actividad cognitiva y control sensorimotor.

In [4]:
# ── HERRAMIENTAS DE PROCESAMIENTO Y CONSULTA EEG ─────────────────────────

# Definición de los rangos de frecuencia estándar para las bandas de interés clínico/educativo (en Hz).
BANDAS_HZ = {
    "Delta": (1.0, 4.0),
    "Theta": (4.0, 8.0),
    "Alpha": (8.0, 13.0),
    "Beta": (13.0, 30.0)
}

def _validar_parametros(fs, duracion=None):
    #Valida las condiciones mínimas para analizar hasta 30 Hz.
    if not np.isfinite(fs) or fs <= 60:
        raise ValueError("La frecuencia de muestreo debe ser mayor que 60 Hz para analizar hasta 30 Hz.")
    if duracion is not None and (not np.isfinite(duracion) or duracion < 2):
        raise ValueError("La duración debe ser de al menos 2 segundos para obtener una resolución frecuencial adecuada.")

def _obtener_senal_eeg(ruta_archivo=None, canal_objetivo="O1", fs=250.0, duracion=4.0, componente_principal="alpha"):
    """
    Obtiene la señal EEG desde un archivo EDF real (usando MNE) o genera una señal
    sintética educativa con una componente sinusoidal dominante.
    Devuelve: señal en microvoltios (uV), frecuencia de muestreo, canal, fuente y advertencias.
    """
    advertencias = []

    if ruta_archivo:
        # Carga y procesamiento de archivo EDF real
        if not os.path.isfile(ruta_archivo):
            raise FileNotFoundError(f"No existe el archivo indicado: {ruta_archivo}")

        raw = mne.io.read_raw_edf(ruta_archivo, preload=True, verbose=False)

        raw.filter(l_freq=1.0, h_freq=None, verbose=False)

        sfreq = float(raw.info["sfreq"])
        _validar_parametros(sfreq)

        # Validación y selección de canales EEG disponibles
        tipos = raw.get_channel_types()
        canales_eeg = [nombre for nombre, tipo in zip(raw.ch_names, tipos) if tipo == "eeg"]
        if not canales_eeg:
            raise ValueError("El archivo EDF no contiene canales identificados por MNE como EEG.")

        canal = canal_objetivo if canal_objetivo in canales_eeg else canales_eeg[0]
        if canal != canal_objetivo:
            advertencias.append(f"El canal solicitado '{canal_objetivo}' no está disponible; se analizó '{canal}' en su lugar.")

        # Extracción de la señal y conversión de Voltios a microvoltios (uV)
        senal = raw.get_data(picks=[canal])[0] * 1e6
        fuente = f"Archivo EDF: {os.path.basename(ruta_archivo)}"
    else:
        # Generación de señal sintética para pruebas pedagógicas
        _validar_parametros(fs, duracion)
        if componente_principal not in {"delta", "theta", "alpha", "beta"}:
            raise ValueError("componente_principal debe ser delta, theta, alpha o beta.")

        sfreq = float(fs)
        t = np.arange(int(round(sfreq * duracion))) / sfreq
        rng = np.random.default_rng(42)
        senal = rng.normal(0.0, 0.5, len(t))

        frecuencias_principales = {"delta": 2.0, "theta": 6.0, "alpha": 10.0, "beta": 20.0}
        amplitudes = {"delta": 3.0, "theta": 3.0, "alpha": 4.0, "beta": 3.0}

        frecuencia = frecuencias_principales[componente_principal]
        amplitud = amplitudes[componente_principal]
        senal += amplitud * np.sin(2 * np.pi * frecuencia * t)

        canal = "Sintético"
        fuente = f"Señal sintética educativa (componente principal: {componente_principal})"

    # Validaciones finales de integridad de la señal
    if senal.ndim != 1 or len(senal) == 0:
        raise ValueError("La señal está vacía o no tiene un único canal.")
    _validar_parametros(sfreq)
    if len(senal) / sfreq < 2:
        raise ValueError("La señal debe tener una duración mínima de 2 segundos.")
    if not np.all(np.isfinite(senal)):
        raise ValueError("La señal contiene valores NaN o infinitos.")

    return senal, sfreq, canal, fuente, advertencias

def _integrar_psd_en_banda(frecuencias, psd, f_min, f_max):
    """Integra la Densidad Espectral de Potencia (PSD) entre dos límites usando la regla trapezoidal."""
    if f_min < frecuencias[0] or f_max > frecuencias[-1]:
        raise ValueError("El espectro no cubre completamente el intervalo solicitado.")

    interiores = (frecuencias > f_min) & (frecuencias < f_max)
    f_banda = np.concatenate(([f_min], frecuencias[interiores], [f_max]))
    p_banda = np.interp(f_banda, frecuencias, psd)

    return float(trapezoid(p_banda, f_banda))

# ── HERRAMIENTA 1: ANÁLISIS ESPECTRAL EEG ────────────────────────────────────
def analizar_senal_eeg(ruta_archivo: str = None, canal_objetivo: str = "O1", fs: float = 250.0, duracion: float = 4.0, componente_principal: str = "alpha") -> str:
    """
    Herramienta 1 del Agente: Analiza un archivo EDF o señal sintética.
    Calcula potencias absolutas y relativas en el rango de 1 a 30 Hz mediante el método de Welch.
    """
    try:
        senal, sfreq, canal, fuente, advertencias = _obtener_senal_eeg(ruta_archivo, canal_objetivo, fs, duracion, componente_principal)

        # Cálculo de la PSD con el método de Welch (ventana de Hann)
        nperseg = min(len(senal), int(round(2 * sfreq)))
        frecuencias, psd = welch(senal, fs=sfreq, window="hann", nperseg=nperseg, noverlap=nperseg // 2, detrend="constant", scaling="density")

        potencia_total = _integrar_psd_en_banda(frecuencias, psd, 1.0, 30.0)
        if not np.isfinite(potencia_total) or potencia_total <= 0:
            raise ValueError("La potencia total entre 1 y 30 Hz es nula o no válida.")

        # Cálculo de potencias absolutas y relativas por banda
        potencias_absolutas = {banda: _integrar_psd_en_banda(frecuencias, psd, limites[0], limites[1]) for banda, limites in BANDAS_HZ.items()}
        potencias_relativas_exactas = {
            banda: (potencia / potencia_total) * 100
            for banda, potencia in potencias_absolutas.items()
        }

        banda_predominante = max(
            potencias_relativas_exactas,
            key=potencias_relativas_exactas.get
        )

        potencias_relativas = {
            banda: round(valor, 2)
            for banda, valor in potencias_relativas_exactas.items()
        }

        # Preparación de datos para la visualización gráfica opcional
        n_tramo = min(len(senal), int(round(4 * sfreq)))
        indices = np.linspace(0, n_tramo - 1, min(n_tramo, 1000), dtype=int)
        frecuencias_vis = frecuencias <= 40.0

        resultado = {
            "fuente": fuente,
            "frecuencia_muestreo_hz": sfreq,
            "canal_analizado": canal,
            "rango_analizado_hz": [1.0, 30.0],
            "bandas_hz": BANDAS_HZ,
            "potencia_total_1_30_hz_uV2": float(potencia_total),
            "potencia_absoluta_uV2": potencias_absolutas,
            "potencia_relativa_porcentaje": potencias_relativas,
            "banda_predominante": banda_predominante,
            "advertencias": advertencias,
            "visualizacion": {
                "tiempo_s": (indices / sfreq).tolist(),
                "senal_uV": senal[indices].tolist(),
                "frecuencias_hz": frecuencias[frecuencias_vis].tolist(),
                "psd_uV2_hz": psd[frecuencias_vis].tolist(),
                "canal": canal,
                "fuente": fuente
            }
        }
        return json.dumps(resultado, ensure_ascii=False)
    except Exception as e:
        return f"ERROR_ANALISIS: {type(e).__name__}: {e}"

# ── HERRAMIENTA 2: CONSULTA DE CONCEPTOS CIENTÍFICOS ────────────────────────
def consultar_conceptos_eeg(concepto: str) -> str:
    """
    Herramienta 2 del Agente: Consulta bandas, artefactos o limitaciones
    en la base de conocimiento local estructurada en JSON.
    """
    try:
        if not isinstance(concepto, str) or not concepto.strip():
            return "ERROR_CONSULTA: Debes indicar un concepto."

        ruta_json = "knowledge_base/conceptos_eeg.json"

        if not os.path.isfile(ruta_json):
            return "ERROR_CONSULTA: No se encuentra knowledge_base/conceptos_eeg.json."

        with open(ruta_json, "r", encoding="utf-8") as f:
            kb = json.load(f)

        # Normalización de acentos y mayúsculas.
        c = unicodedata.normalize("NFKD", concepto.casefold().strip())
        c = "".join(
            caracter for caracter in c
            if not unicodedata.combining(caracter)
        )

        sinonimos = {"alfa": "alpha"}

        for sinonimo, forma_estandar in sinonimos.items():
            c = re.sub(rf"\b{sinonimo}\b", forma_estandar, c)

        resultado = {}

        bandas_encontradas = [
            banda for banda in kb["bandas"]
            if re.search(rf"\b{re.escape(banda)}\b", c)
        ]

        if bandas_encontradas:
            resultado["bandas"] = {
                banda.upper(): kb["bandas"][banda]
                for banda in bandas_encontradas
            }

        if any(p in c for p in ["artefacto", "ruido", "emg", "eog"]):
            resultado["artefactos"] = kb["artefactos"]

        if any(p in c for p in [
            "limitacion", "diagnostico", "epilepsia", "clinico"
        ]):
            resultado["limitaciones"] = kb["limitaciones"]

        if resultado:
            return json.dumps(resultado, ensure_ascii=False)

        return (
            f"CONCEPTO_NO_ENCONTRADO: '{c}' "
            "no está en la base de conocimiento local."
        )

    except Exception as e:
        return f"ERROR_CONSULTA: {type(e).__name__}: {e}"


# ── HERRAMIENTA 3: COMPARACIÓN Y VALIDACIÓN DE BANDAS ───────────────────────
def comparar_bandas(datos_json: str) -> str:
    """
    Herramienta 3 del Agente: Valida y compara las cuatro potencias relativas
    devueltas por el análisis espectral.
    """
    try:
        data = json.loads(datos_json)
        potencias = data.get("potencia_relativa_porcentaje", {})
        bandas_requeridas = ["Delta", "Theta", "Alpha", "Beta"]

        if not all(banda in potencias for banda in bandas_requeridas):
            return "ERROR_COMPARACION: Se requieren las potencias relativas de Delta, Theta, Alpha y Beta."

        valores = {b: float(potencias[b]) for b in bandas_requeridas}
        if any(not np.isfinite(v) or v < 0 or v > 100 for v in valores.values()):
            return "ERROR_COMPARACION: Las potencias deben ser números entre 0 y 100."

        suma = sum(valores.values())
        if not np.isclose(suma, 100.0, atol=0.15):
            return f"ERROR_COMPARACION: Las potencias relativas no suman ~100%. Suma: {suma:.2f}%."

        mayor = max(valores.values())
        predominantes = [b for b, v in valores.items() if np.isclose(v, mayor)]
        diferencia = max(valores.values()) - min(valores.values())

        lineas = [f"Fuente analizada: {data.get('fuente', 'No especificada')}", "Potencias relativas:"]
        lineas.extend([f"- {b}: {valores[b]:.2f} %" for b in bandas_requeridas])

        if len(predominantes) == 1:
            lineas.append(f"La banda predominante es {predominantes[0]} ({mayor:.2f} %).")
        else:
            lineas.append(f"Hay un empate en la mayor potencia relativa: {', '.join(predominantes)} ({mayor:.2f} %).")

        lineas.extend([
            f"La diferencia entre la mayor y la menor potencia relativa es {diferencia:.2f} puntos porcentuales.",
            "La predominancia describe la distribución espectral de esta señal; no constituye por sí sola una interpretación clínica."
        ])
        return "\n".join(lineas)
    except Exception as e:
        return f"ERROR_COMPARACION: {type(e).__name__}: {e}"

# ── REGISTRO DE HERRAMIENTAS Y ESPECIFICACIÓN JSON PARA EL LLM (NVIDIA NIM) ──
TOOL_REGISTRY = {
    "analizar_senal_eeg": analizar_senal_eeg,
    "consultar_conceptos_eeg": consultar_conceptos_eeg,
    "comparar_bandas": comparar_bandas,
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "analizar_senal_eeg",
            "description": "Analiza una señal EEG desde un archivo EDF o genera una señal sintética. Devuelve potencias en Delta, Theta, Alpha y Beta.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ruta_archivo": {"type": "string", "description": "Ruta local de un archivo EDF. Omítela para generar una señal sintética."},
                    "canal_objetivo": {"type": "string", "description": "Canal EEG solicitado en el EDF. Por defecto O1."},
                    "fs": {"type": "number", "description": "Frecuencia de muestreo en Hz para la señal sintética (>60)."},
                    "duracion": {"type": "number", "description": "Duración en segundos para la señal sintética (>=2)."},
                    "componente_principal": {"type": "string", "enum": ["delta", "theta", "alpha", "beta"], "description": "Banda sinusoidal principal de la señal sintética."}
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "consultar_conceptos_eeg",
            "description": "Consulta la base de conocimiento local sobre una o varias bandas EEG, artefactos o limitaciones.",
            "parameters": {
                "type": "object",
                "properties": {
                    "concepto": {"type": "string", "description": "Concepto o bandas a consultar (ej: 'theta y delta', 'alpha')."}
                },
                "required": ["concepto"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "comparar_bandas",
            "description": "Compara las potencias relativas de las cuatro bandas devueltas por analizar_senal_eeg.",
            "parameters": {
                "type": "object",
                "properties": {
                    "datos_json": {"type": "string", "description": "JSON completo devuelto por analizar_senal_eeg."}
                },
                "required": ["datos_json"]
            }
        }
    }
]

print("Herramientas EEG definidas correctamente.")

Herramientas EEG definidas correctamente.


# **Configuración del agente**

## **Integración con el modelo de lenguaje**

El agente se conecta al modelo `nvidia/nemotron-3.5-super-vl-120b-a12b` a través de un cliente compatible con la API de OpenAI (`https://integrate.api.nvidia.com/v1`).

Para gestionar las consultas, implementa un **bucle iterativo de llamada a herramientas (*function calling*)** similar al visto en clase, pero con ajustes para mantener el historial conversacional de la interfaz en Gradio, extraer datos para las gráficas y controlar un límite máximo de tres ejecuciones que permita generar respuestas parciales sin interrupciones abruptas.

In [5]:
# ── BUCLE DE EJECUCIÓN DEL AGENTE Y GESTIÓN DE HERRAMIENTAS ───────────────
def run_agent(user_message: str, history=None, llamadas_previas: int = 0):
    """
    Ejecuta el ciclo de razonamiento y llamadas a herramientas (agentic loop) para el agente EEG.

    Gestiona un límite estricto de hasta tres llamadas reales a herramientas por consulta.
    Si se alcanza el límite, procesa la información obtenida hasta ese momento y evita alucinaciones.

    Devuelve:
        Una tupla con (respuesta_texto, lista_datos_visualizacion).
    """
    # ── 1. DEFINICIÓN DEL SYSTEM PROMPT Y DIRECTRICES DE COMPORTAMIENTO ───────
    # Establece la identidad pedagógica, las reglas científicas, de prudencia clínica
    # y la gestión transparente del límite operativo de llamadas a herramientas.
    system_prompt = (
        "Eres Asistente-EEG, un asistente educativo sobre electroencefalografía. "
        "Responde siempre en español, con rigor y claridad.\n\n"

        "No diagnostiques enfermedades ni infieras trastornos del sueño o estados psicológicos. "
        "Distingue los resultados numéricos de su interpretación y usa expresiones prudentes.\n"

        "Para analizar señales nuevas utiliza analizar_senal_eeg. Para consultas conceptuales utiliza "
        "consultar_conceptos_eeg. Para resumir las bandas de una señal puedes utilizar comparar_bandas, "
        "pero solo después de recibir el JSON del análisis; no solicites ambas herramientas en el mismo lote.\n"

        "Si recibes resultados precalculados de archivos EDF, utilízalos y no vuelvas a analizar esos archivos. "
        "Si comparas varias señales cuyos resultados ya están disponibles, compara directamente sus valores: "
        "comparar_bandas resume las bandas de una señal, no compara dos archivos distintos.\n"

        "La potencia relativa se calcula respecto a la potencia total entre 1 y 30 Hz. "
        "Las bandas son Delta [1,4), Theta [4,8), Alpha [8,13) y Beta [13,30] Hz.\n"

        "El sistema solo calcula esas cuatro bandas. Si preguntan por Gamma u otra banda fuera de ese rango, "
        "indica que no está implementada y no inventes valores.\n"

        "Las señales sintéticas son educativas y no reproducen toda la complejidad de un EEG real.\n\n"

        "Dispones de un máximo de tres llamadas reales a herramientas por consulta, contando los análisis "
        "de archivos EDF ejecutados antes de entrar al agente. Ejecuta todas las llamadas que todavía quepan. "
        "Las que excedan el límite deben quedar marcadas como NO_EJECUTADA. Nunca inventes sus resultados. "
        "Después de alcanzar el límite, redacta una respuesta final con los resultados disponibles e indica "
        "qué quedó pendiente. No afirmes que completaste tareas no ejecutadas."
    )

    # ── 2. CONSTRUCCIÓN DEL HISTORIAL DE MENSAJES ───────────────────────────
    # Inicializa el array de mensajes con el contexto del sistema.
    messages = [{"role": "system", "content": system_prompt}]

    for item in (history or []):
        if isinstance(item, dict) and item.get("role") in {"user", "assistant"}:
            contenido = item.get("content")
            if isinstance(contenido, str) and contenido:
                messages.append({"role": item["role"], "content": contenido})

        elif isinstance(item, (list, tuple)) and len(item) == 2:
            if item[0]:
                messages.append({"role": "user", "content": str(item[0])})
            if item[1]:
                messages.append({"role": "assistant", "content": str(item[1])})

    # Añade el mensaje actual del usuario al final de la cola.
    messages.append({"role": "user", "content": user_message})

    # Control de contadores y validación inicial de llamadas previas.
    llamadas_realizadas = int(llamadas_previas)
    if llamadas_realizadas < 0 or llamadas_realizadas > 3:
        raise ValueError("llamadas_previas debe estar entre 0 y 3.")

    datos_visualizacion = []
    forzar_respuesta_final = llamadas_realizadas >= 3

    # ── 3. BUCLE DE RAZONAMIENTO E ITERACIÓN (AGENTIC LOOP) ───────────────────
    while True:
        # Petición al modelo LLM a través de la API de NVIDIA NIM.
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="none" if forzar_respuesta_final else "auto"
        )

        choice = response.choices[0]
        assistant_msg = choice.message

        # Si el modelo decide terminar sin llamar a herramientas, retorna la respuesta generada.
        if choice.finish_reason == "stop":
            return (
                assistant_msg.content or "No se generó una respuesta.",
                datos_visualizacion
            )

        # Validación en caso de que la respuesta no sea una llamada válida a herramientas.
        if choice.finish_reason != "tool_calls" or not assistant_msg.tool_calls:
            return (
                assistant_msg.content or "El modelo no pudo completar la consulta.",
                datos_visualizacion
            )

        # Si el proveedor ignora tool_choice="none", no se ejecutan más herramientas.
        if forzar_respuesta_final:
            return (
                assistant_msg.content
                or "Se alcanzó el límite de herramientas. Los resultados son parciales.",
                datos_visualizacion
            )

        # Cada ID de llamada debe recibir una respuesta, incluso si queda pendiente.
        messages.append(assistant_msg.model_dump(exclude_none=True))

        hay_analisis_en_lote = any(
            tc.function.name == "analizar_senal_eeg"
            for tc in assistant_msg.tool_calls
        )

        hubo_llamadas_pendientes = False

        # ── 4. EJECUCIÓN DINÁMICA DE HERRAMIENTAS ────────────────────────────
        for tc in assistant_msg.tool_calls:
            tool_name = tc.function.name

            if llamadas_realizadas >= 3:
                resultado = (
                    "NO_EJECUTADA: se alcanzó el límite de tres llamadas. "
                    "La tarea queda pendiente; no se generaron resultados."
                )
                hubo_llamadas_pendientes = True

            elif tool_name == "comparar_bandas" and hay_analisis_en_lote:
                resultado = (
                    "NO_EJECUTADA: comparar_bandas depende del JSON de analizar_senal_eeg. "
                    "Espera el resultado del análisis y solicita la comparación después."
                )

            else:
                # Cada invocación despachada consume un turno, incluso si devuelve un error.
                llamadas_realizadas += 1

                try:
                    tool_args = json.loads(tc.function.arguments)
                    fn = TOOL_REGISTRY.get(tool_name)

                    if fn is None:
                        resultado = f"ERROR: herramienta '{tool_name}' no encontrada."
                    else:
                        resultado = fn(**tool_args)

                    # Extracción automática de metadatos de visualización si la herramienta es de análisis EEG.
                    if tool_name == "analizar_senal_eeg" and isinstance(resultado, str):
                        try:
                            datos = json.loads(resultado)
                            visualizacion = datos.pop("visualizacion", None)

                            if visualizacion:
                                datos_visualizacion.append(visualizacion)

                            resultado = json.dumps(datos, ensure_ascii=False)

                        except json.JSONDecodeError:
                            pass

                except Exception as e:
                    resultado = (
                        f"ERROR_EJECUCION_HERRAMIENTA: {type(e).__name__}: {e}"
                    )

            # Traza de la herramienta (nombre, argumentos y resultado) para depuración y documentación de pruebas.
            print(
                f"[TOOL | llamadas acumuladas: {llamadas_realizadas}/3] {tool_name}\n"
                f"  argumentos: {tc.function.arguments}\n"
                f"  resultado: {str(resultado)[:1000]}\n"
            )

            # Añade el resultado de la ejecución de la herramienta al stack de mensajes del LLM.
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(resultado)})

        if hubo_llamadas_pendientes or llamadas_realizadas >= 3:
            forzar_respuesta_final = True


print("Agente configurado: máximo de tres llamadas y respuestas parciales.")


Agente configurado: máximo de tres llamadas y respuestas parciales.


---

# **Casos de prueba y validación**

Para validar el funcionamiento del agente, la selección adecuada de herramientas y el respeto a sus restricciones operativas, se evaluaron **seis escenarios de prueba** conforme a los criterios del proyecto y casos de estrés operativo:

1. **Caso 1 — Análisis de señal sintética Alpha (Habitual 1):** Generación de una señal controlada con componente Alpha, cálculo de potencias absolutas y relativas e identificación de predominancia (`analizar_senal_eeg`).
2. **Caso 2 — Consulta conceptual sobre Alpha (Habitual 2):** Explicación del fenómeno de atenuación de la actividad Alpha al abrir los ojos (reactividad Alpha) y sus límites interpretativos (`consultar_conceptos_eeg`).
3. **Caso 3 — Comparación de dos señales sintéticas (Habitual 3):** Generación y análisis paralelo de señales Delta y Beta para contrastar sus distribuciones espectrales (`analizar_senal_eeg`).
4. **Caso 4 — Análisis integral de señal sintética Theta y consulta conceptual (Uso combinado de herramientas):** Generación de una señal Theta, comparación de bandas y explicación de sus características neurofisiológicas dentro del rango implementado (`analizar_senal_eeg`, `comparar_bandas`, `consultar_conceptos_eeg`).
5. **Caso 5 — Consulta fuera del alcance y de la base de conocimiento (Caso límite):** Solicitud combinada sobre la banda Gamma (concepto teórico externo y cálculo de porcentaje en señal generada) para verificar que el agente recurra a su conocimiento base, respete las restricciones operativas ($1 - 30\text{ Hz}$) y se niegue a inventar métricas numéricas.
6. **Caso 6 — Solicitud masiva y gestión del límite de llamadas (Caso adicional):** Solicitud de generación de 10 señales para comprobar el respeto estricto al tope operativo de 3 llamadas por consulta, informando tareas pendientes (`NO_EJECUTADAS`) sin alucinar resultados.

> **Documentación detallada:** Para revisar los prompts exactos, las tablas de referencia numérica, las respuestas completas obtenidas y el análisis de limitaciones y mejoras, consulta el archivo **[README.md en el repositorio de GitHub](https://github.com/LotAlejandroV/Asistente_EEG/blob/main/README.md)**.

---

### **Verificación rápida del bucle del agente**

Antes de iniciar la interfaz web, se ejecutan dos consultas de prueba directamente sobre `run_agent` para evidenciar la selección de herramientas, los argumentos enviados y las respuestas generadas por el modelo.

In [6]:
# ── EVIDENCIA DE EJECUCIÓN: CASO 1 (ANÁLISIS DE SEÑAL SINTÉTICA) ───────────
pregunta_caso_1 = (
    "Genera una señal sintética con componente principal Alpha. "
    "Analiza sus potencias absolutas y relativas e identifica la banda predominante."
)

respuesta_1, _ = run_agent(pregunta_caso_1)
print("── RESPUESTA FINAL DEL AGENTE ──")
print(respuesta_1)

[TOOL | llamadas acumuladas: 1/3] analizar_senal_eeg
  argumentos: {"componente_principal":"alpha","fs":256,"duracion":10}
  resultado: {"fuente": "Señal sintética educativa (componente principal: alpha)", "frecuencia_muestreo_hz": 256.0, "canal_analizado": "Sintético", "rango_analizado_hz": [1.0, 30.0], "bandas_hz": {"Delta": [1.0, 4.0], "Theta": [4.0, 8.0], "Alpha": [8.0, 13.0], "Beta": [13.0, 30.0]}, "potencia_total_1_30_hz_uV2": 8.052759236757275, "potencia_absoluta_uV2": {"Delta": 0.006443472199284321, "Theta": 0.007894292799730324, "Alpha": 8.002056923461039, "Beta": 0.03636454829722163}, "potencia_relativa_porcentaje": {"Delta": 0.08, "Theta": 0.1, "Alpha": 99.37, "Beta": 0.45}, "banda_predominante": "Alpha", "advertencias": []}

── RESPUESTA FINAL DEL AGENTE ──
A continuación se presentan los resultados del análisis de la señal sintética generada con componente principal **Alpha**.

### Parámetros de la señal
- **Fuente:** Señal sintética educativa (componente principal: alpha)

In [7]:
# ── EVIDENCIA DE EJECUCIÓN: CASO 4 (BASE DE CONOCIMIENTO DE ARTEFACTOS) ────
pregunta_caso_4 = (
    "¿Qué artefactos EOG y EMG pueden contaminar una señal EEG? "
    "Explica cómo pueden afectar las frecuencias bajas y la banda Beta."
)

respuesta_4, _ = run_agent(pregunta_caso_4)
print("── RESPUESTA FINAL DEL AGENTE ──")
print(respuesta_4)

[TOOL | llamadas acumuladas: 1/3] consultar_conceptos_eeg
  argumentos: {"concepto":"artefactos EOG EMG"}
  resultado: {"artefactos": {"descripcion": "Señales no cerebrales o interferencias instrumentales que se superponen al EEG y pueden modificar su espectro.", "tipos_frecuentes": ["Oculares (EOG): parpadeos y movimientos oculares producen deflexiones de gran amplitud, especialmente en electrodos frontales, con contribución importante a bajas frecuencias.", "Musculares (EMG): la actividad de músculos faciales, mandibulares y cervicales puede introducir potencia de alta frecuencia, incluida la banda Beta.", "Instrumentales y de red: mal contacto de electrodos, movimiento de cables e interferencia eléctrica de 50 o 60 Hz."]}}

── RESPUESTA FINAL DEL AGENTE ──
Los artefactos son señales no cerebrales o interferencias instrumentales que se superponen al registro EEG y pueden modificar su espectro de frecuencias. A continuación se describen los dos tipos solicitados y su posible influenci

---
# **Interfaz de usuario**

### **Interfaz interactiva**

El sistema incorpora una interfaz web desarrollada con **Gradio** (`gr.Blocks`), diseñada para facilitar la interacción directa con el agente educativo sin necesidad de modificar código. Desde este entorno, el usuario puede:

* **Formular preguntas en lenguaje natural** sobre conceptos neurofisiológicos o solicitar análisis de señales.
* **Cargar hasta dos archivos EEG reales** en formato `.edf` de forma simultánea para realizar evaluaciones individuales o comparativas.
* **Visualizar las señales y sus espectros**, mostrando en paralelo la forma de onda en el dominio del tiempo ($\mu\text{V}$) y su Densidad Espectral de Potencia ($\text{PSD}$) mediante el método de Welch.
* **Recibir explicaciones fundamentadas** y contextualizadas a partir de los resultados numéricos obtenidos tras el procesamiento.

In [8]:
# ── INTERFAZ GRÁFICA Y VISUALIZACIÓN (GRADIO) ─────────────────────────────
def generar_grafica_eeg(datos_visualizacion):
    """Grafica la misma señal analizada: forma de onda temporal y densidad espectral de potencia."""
    if not datos_visualizacion:
        return None

    tiempo = np.asarray(datos_visualizacion["tiempo_s"], dtype=float)
    senal = np.asarray(datos_visualizacion["senal_uV"], dtype=float)
    frecuencias = np.asarray(datos_visualizacion["frecuencias_hz"], dtype=float)
    psd = np.asarray(datos_visualizacion["psd_uV2_hz"], dtype=float)
    canal = datos_visualizacion.get("canal", "No especificado")
    fuente = datos_visualizacion.get("fuente", "Señal EEG")

    if len(tiempo) == 0 or len(senal) != len(tiempo) or len(frecuencias) == 0 or len(psd) != len(frecuencias):
        return None

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6))
    fig.suptitle(f"Visualización EEG — {canal}", fontsize=14)

    # ── Dominio temporal ──────────────────────────────────
    ax1.plot(tiempo, senal, linewidth=0.9)
    ax1.set_title("Señal en el dominio del tiempo (primeros 4 s)")
    ax1.set_xlabel("Tiempo (s)")
    ax1.set_ylabel("Amplitud (µV)")
    ax1.grid(True, alpha=0.3)

    # ── Dominio frecuencial ──────────────────────────────-
    positivos = psd > 0
    if np.any(positivos):
        ax2.semilogy(frecuencias[positivos], psd[positivos], linewidth=1.5)

    bandas_grafica = [("Delta", 1, 4), ("Theta", 4, 8), ("Alpha", 8, 13), ("Beta", 13, 30)]
    for nombre, f_min, f_max in bandas_grafica:
        ax2.axvline(f_min, linestyle=":", linewidth=0.8, alpha=0.6)
        ax2.axvline(f_max, linestyle=":", linewidth=0.8, alpha=0.6)

    ax2.set_xlim(1, 30)
    ax2.set_title("Densidad espectral de potencia (Welch)")
    ax2.set_xlabel("Frecuencia (Hz)")
    ax2.set_ylabel("PSD (µV²/Hz)")
    ax2.grid(True, alpha=0.3)

    fig.text(0.01, 0.01, fuente, fontsize=8)
    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    return fig

def _rutas_archivos_gradio(archivos_subidos):
    """
    Normaliza el objeto devuelto por el componente gr.File de Gradio
    (cuando file_count="multiple") para extraer de forma segura una lista de rutas locales.
    """
    if archivos_subidos is None:
        return []

    # Si Gradio devuelve un único archivo o una lista dependiendo de la versión
    if not isinstance(archivos_subidos, list):
        archivos_subidos = [archivos_subidos]

    rutas = []
    for archivo in archivos_subidos:
        if isinstance(archivo, str):
            rutas.append(archivo)
        elif hasattr(archivo, "name"):
            rutas.append(archivo.name)

    return rutas

def responder_con_interaccion_y_grafica(message, archivos_subidos, history):
    """
    Función controladora de la interfaz de Gradio adaptada para hasta 2 archivos.
    """
    history = list(history or [])
    figura_1, figura_2 = None, None
    prompt = (message or "").strip()

    try:
        rutas = _rutas_archivos_gradio(archivos_subidos)

        # Validación estricta: Máximo 2 archivos
        if len(rutas) > 2:
            respuesta = "⚠️ Solo se permite cargar un máximo de 2 archivos simultáneamente. Por favor, selecciona solo dos."
            history.extend([{"role": "user", "content": prompt or "Cargar archivos."}, {"role": "assistant", "content": respuesta}])
            return history, "", None, None

        visualizaciones = []
        contexto_archivos = ""

        # ── 1. PRE-ANÁLISIS DE LOS ARCHIVOS EDF CARGADOS ─────────────────────
        for i, ruta in enumerate(rutas, start=1):
            resultado_archivo = analizar_senal_eeg(ruta_archivo=ruta, canal_objetivo="O1")

            if resultado_archivo.startswith("ERROR_ANALISIS:"):
                respuesta = f"No fue posible analizar el archivo {i} cargado.\n\n" + resultado_archivo
                history.extend([{"role": "user", "content": prompt or "Analiza los archivos cargados."}, {"role": "assistant", "content": respuesta}])
                return history, "", None, None

            datos_archivo = json.loads(resultado_archivo)
            visual_archivo = datos_archivo.pop("visualizacion", None)

            if visual_archivo:
                visualizaciones.append(visual_archivo)

            contexto_archivos += f"\n\n[Contexto del sistema - Archivo {i} ({os.path.basename(ruta)}): " + json.dumps(datos_archivo, ensure_ascii=False) + "]"

        if contexto_archivos:
            prompt += "\n\n[Contexto del sistema: los archivos EDF cargados ya fueron analizados previamente con las herramientas." + contexto_archivos + "]"

        if not prompt and not rutas:
            return history, "", None, None

        # ── 2. EJECUCIÓN DEL CICLO DEL AGENTE ────────────────────────────────
        # Se ajustan las llamadas previas según la cantidad de archivos analizados externamente
        respuesta, visual_agente = run_agent(prompt, history, llamadas_previas=len(rutas))

        if visual_agente:
            visualizaciones.extend(visual_agente)

        # Generación de las figuras gráficas de Matplotlib para los 2 paneles
        if len(visualizaciones) >= 1:
            figura_1 = generar_grafica_eeg(visualizaciones[0])
        if len(visualizaciones) >= 2:
            figura_2 = generar_grafica_eeg(visualizaciones[1])
        if len(visualizaciones) > 2:
            respuesta += "\n\nNota: se procesaron más de dos señales, pero la interfaz muestra únicamente las dos primeras."

        # Actualiza el historial con el nuevo turno
        history.extend([{"role": "user", "content": message or "Analiza los archivos cargados."}, {"role": "assistant", "content": respuesta}])
        return history, "", figura_1, figura_2

    except Exception as e:
        respuesta = f"Se produjo un error al procesar la consulta: {type(e).__name__}: {e}"
        history.extend([{"role": "user", "content": message or "Procesar señales EEG."}, {"role": "assistant", "content": respuesta}])
        return history, "", figura_1, figura_2

# ── 3. CONSTRUCCIÓN DE LA APLICACIÓN WEB CON GRADIO BLOCKS ───────────────────
with gr.Blocks() as demo:
    gr.Markdown("# 🧠 Asistente-EEG: análisis educativo de señales EEG")
    gr.Markdown("Consulta conceptos neurofisiológicos o carga **hasta dos archivos EDF** para realizar análisis comparativos.")

    with gr.Row():
        # Columna izquierda: Chatbot y entrada de texto
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label="Chat educativo")
            mensaje_input = gr.Textbox(placeholder="Ej.: Compara las bandas predominantes de ambos archivos...", label="Mensaje", container=False)
            btn_enviar = gr.Button("Enviar", variant="primary")

        # Columna derecha: Carga de archivos múltiples y paneles de visualización
        with gr.Column(scale=2):
            file_input = gr.File(
                label="Cargar archivos EDF (Máximo 2)",
                file_types=[".edf"],
                file_count="multiple",  # <--- Habilita la selección de múltiples archivos
                type="filepath"
            )
            gr.Markdown("### Visualizaciones EEG (Comparativa)")

            with gr.Row():
                with gr.Column(): plot_output_1 = gr.Plot(label="Señal / Archivo 1")
                with gr.Column(): plot_output_2 = gr.Plot(label="Señal / Archivo 2")
            gr.Markdown("Las visualizaciones se actualizan automáticamente al procesar las señales.")

    # ── 4. VINCULACIÓN DE EVENTOS ────────────────────────────────────────────
    btn_enviar.click(fn=responder_con_interaccion_y_grafica, inputs=[mensaje_input, file_input, chatbot], outputs=[chatbot, mensaje_input, plot_output_1, plot_output_2])
    mensaje_input.submit(fn=responder_con_interaccion_y_grafica, inputs=[mensaje_input, file_input, chatbot], outputs=[chatbot, mensaje_input, plot_output_1, plot_output_2])

demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://394558fd18f45baa1e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://394558fd18f45baa1e.gradio.live
